In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# set the themes
sns.set_theme(style="whitegrid")
%matplotlib inline

# Read the csv file

In [2]:
df = pd.read_csv('data/co2_data.csv')

# Tailor the dataframe

Since the dataframe is pretty large it might be convenient to remove all the unnecessary columns. We remove the columns and copy the df to keep the original dataframe. 

Then we remove continents and other subgroups, which could screw with our analysis later. Entries without iso-code get removed by .notna() or checking for empty strings.


In [3]:
# choose what columns to keep (can be changed, but this is the most important)
selected_columns = ['country', 'year', 'iso_code', 'population', 'co2']

# drop the columns that are not in the selected_columns list
df = df[selected_columns].copy()

# remove entries with no iso code(Continents and other groups)
df = df[df['iso_code'].notna() & (df['iso_code'].str.strip() != '')]



# Load and prepare GDP data

In [4]:
# load GDP data
gdp_df = pd.read_csv('data/gdp_data.csv')

# reshape from wide to long format
# melt the year columns into rows
year_columns = [col for col in gdp_df.columns if col.isdigit()]
gdp_long = gdp_df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_columns,
    var_name='year',
    value_name='gdp'
)

# convert year to integer and gdp to numeric
gdp_long['year'] = gdp_long['year'].astype(int)
gdp_long['gdp'] = pd.to_numeric(gdp_long['gdp'], errors='coerce')

# rename Country Code to iso_code for merging
gdp_long = gdp_long.rename(columns={'Country Code': 'iso_code'})

# keep only the columns we need
gdp_long = gdp_long[['iso_code', 'year', 'gdp']]

# remove rows with missing GDP values
gdp_long = gdp_long.dropna(subset=['gdp'])

print(f"GDP data shape: {gdp_long.shape}")
gdp_long.head()

GDP data shape: (14561, 3)


,iso_code,year,gdp
1,AFE,1960,2.420569e+10
3,AFW,1960,1.190481e+10
9,ARG,1960,1.586547e+10
13,AUS,1960,1.863568e+10
14,AUT,1960,6.624086e+09


# Merge CO2 and GDP datasets

We'll merge the two datasets on `iso_code` and `year`. This will combine the CO2 emissions data with GDP data for each country-year combination.

In [5]:
# merge the datasets on iso_code and year
merged_df = df.merge(
    gdp_long,
    on=['iso_code', 'year'],
    how='left',  # keep all CO2 data, add GDP where available
    suffixes=('_co2', '_gdp')
)

# check the merge results
print(f"Original CO2 data shape: {df.shape}")
print(f"GDP data shape: {gdp_long.shape}")
print(f"Merged data shape: {merged_df.shape}")
print(f"\nNumber of rows with GDP data: {merged_df['gdp'].notna().sum()}")
print(f"Number of rows without GDP data: {merged_df['gdp'].isna().sum()}")

# display first few rows
merged_df.head(10)

Original CO2 data shape: (42480, 5)
GDP data shape: (14561, 3)
Merged data shape: (42480, 6)

Number of rows with GDP data: 11317
Number of rows without GDP data: 31163


,country,year,iso_code,population,co2,gdp
0,Afghanistan,1750,AFG,2802560.0,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,NaN
5,Afghanistan,1755,AFG,NaN,NaN,NaN
6,Afghanistan,1756,AFG,NaN,NaN,NaN
7,Afghanistan,1757,AFG,NaN,NaN,NaN
8,Afghanistan,1758,AFG,NaN,NaN,NaN
9,Afghanistan,1759,AFG,NaN,NaN,NaN


# Analysis ready!

The `merged_df` dataframe now contains both CO2 emissions and GDP data. You can now analyze the relationship between economic activity (GDP) and carbon emissions.